In [ ]:
import json, os, urllib.request
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

OUT = "output"; os.makedirs(OUT, exist_ok=True); os.makedirs(f"{OUT}/charts", exist_ok=True)
RAW_PATH = "online_retail_raw.csv"

## 1. LOAD

In [ ]:
# ---------- 1. LOAD ----------
if not os.path.exists(RAW_PATH):
    import pyreadr
    urllib.request.urlretrieve("https://raw.githubusercontent.com/allanvc/onlineretail/HEAD/data/onlineretail.rda", "or.rda")
    pyreadr.read_r("or.rda")["onlineretail"].to_csv(RAW_PATH, index=False)
raw = pd.read_csv(RAW_PATH, parse_dates=["InvoiceDate"], dtype={"InvoiceNo": str, "StockCode": str})
R = {"raw_rows": len(raw), "raw_cols": raw.shape[1]}

## 2. CLEAN

In [ ]:
# ---------- 2. CLEAN ----------
df = raw.drop_duplicates().copy()
R["dup_rows"] = len(raw) - len(df)
df["Description"] = df["Description"].str.strip().str.upper()
df["Revenue"] = (df["Quantity"] * df["UnitPrice"]).round(2)
df["IsCancel"] = df["InvoiceNo"].str.startswith("C")
NONPROD = ["POST","DOT","M","C2","D","S","BANK CHARGES","AMAZONFEE","CRUK","B","PADS","m"]
df["NonProduct"] = (df.StockCode.isin(NONPROD) | df.StockCode.str.lower().str.startswith("gift_")
                    | df.StockCode.str.startswith("DCGS"))
R["missing_desc"] = int(df.Description.isna().sum())
R["missing_cust_rows"] = int(df.CustomerID.isna().sum())
R["cancel_rows"] = int(df.IsCancel.sum())
R["nonproduct_rows"] = int(df.NonProduct.sum())
R["zero_price_rows"] = int((df.UnitPrice <= 0).sum())

# Cancelled-in-full outlier orders (same customer, same quantity, cancelled minutes later)
OUTLIER_INV = ["581483", "541431"]; OUTLIER_CANCEL = ["C581484", "C541433"]

sales = df[(~df.IsCancel) & (df.Quantity > 0) & (df.UnitPrice > 0) & (~df.NonProduct)
           & (~df.InvoiceNo.isin(OUTLIER_INV))].copy()
rets = df[(df.IsCancel) & (df.UnitPrice > 0) & (~df.NonProduct)
          & (~df.InvoiceNo.isin(OUTLIER_CANCEL))].copy()
R["clean_sales_rows"] = len(sales); R["return_rows"] = len(rets)

sales["Month"] = sales.InvoiceDate.dt.to_period("M").astype(str)
sales["Weekday"] = sales.InvoiceDate.dt.day_name()
sales["Hour"] = sales.InvoiceDate.dt.hour
sales["CustomerType"] = np.where(sales.CustomerID.isna(), "Guest (no ID)", "Identified")
sales["Region"] = np.where(sales.Country == "United Kingdom", "UK", "Export")
sales["CustomerID"] = sales.CustomerID.astype("Int64")

## 3. RFM (identified customers only)

In [ ]:
# ---------- 3. RFM (identified customers only) ----------
idn = sales[sales.CustomerID.notna()]
snap = sales.InvoiceDate.max() + pd.Timedelta(days=1)
rfm = idn.groupby("CustomerID").agg(last=("InvoiceDate","max"), freq=("InvoiceNo","nunique"), mon=("Revenue","sum"))
rfm["recency"] = (snap - rfm["last"]).dt.days
rfm["R"] = pd.qcut(rfm.recency, 4, labels=[4,3,2,1]).astype(int)
rfm["F"] = pd.qcut(rfm.freq.rank(method="first"), 4, labels=[1,2,3,4]).astype(int)
rfm["M"] = pd.qcut(rfm.mon.rank(method="first"), 4, labels=[1,2,3,4]).astype(int)
def seg(x):
    if x.R >= 3 and x.F >= 3 and x.M >= 3: return "Champions"
    if x.R >= 3 and x.F >= 2: return "Loyal / Active"
    if x.R <= 2 and x.F >= 3: return "At Risk (valuable, gone quiet)"
    if x.R == 4: return "New / Recent"
    return "Lapsed / Low value"
rfm["Segment"] = rfm.apply(seg, axis=1)
sales = sales.merge(rfm[["Segment"]], left_on="CustomerID", right_index=True, how="left")
sales["Segment"] = sales.Segment.fillna("Guest (no ID)")

## 4. PROCESSED DATA (invoice level, analysis-ready)

In [ ]:
# ---------- 4. PROCESSED DATA (invoice level, analysis-ready) ----------
inv = sales.groupby("InvoiceNo").agg(
    InvoiceDate=("InvoiceDate","first"), Month=("Month","first"), Weekday=("Weekday","first"), Hour=("Hour","first"),
    CustomerID=("CustomerID","first"), CustomerType=("CustomerType","first"), Segment=("Segment","first"),
    Country=("Country","first"), Region=("Region","first"),
    ProductLines=("StockCode","nunique"), Units=("Quantity","sum"), Revenue=("Revenue","sum")).reset_index()
inv["Revenue"] = inv.Revenue.round(2)
inv["OrderSize"] = pd.cut(inv.Revenue, [0,250,1000,1e9], labels=["Small (<250)","Medium (250-1000)","Large (>1000)"])
inv.to_csv(f"{OUT}/processed_data_invoice_level.csv", index=False)
sales.to_csv(f"{OUT}/processed_lines_clean.csv", index=False)

## 5. ANALYSIS

In [ ]:
# ---------- 5. ANALYSIS ----------
gross = sales.Revenue.sum(); ret_val = -rets.Revenue.sum()
R.update(gross=gross, returns=ret_val, net=gross-ret_val, return_rate=ret_val/gross,
         orders=int(inv.shape[0]), identified_customers=int(idn.CustomerID.nunique()), aov=gross/len(inv),
         median_order=float(inv.Revenue.median()), products=int(sales.StockCode.nunique()))
R["guest_rev_share"] = sales[sales.CustomerID.isna()].Revenue.sum()/gross
R["guest_line_share"] = float(sales.CustomerID.isna().mean())

mon = inv.groupby("Month").agg(Revenue=("Revenue","sum"), Orders=("InvoiceNo","count"), Customers=("CustomerID","nunique"))
mon["AOV"] = mon.Revenue/mon.Orders; mon.to_csv(f"{OUT}/summary_monthly.csv")
R["sepnov_share"] = mon.loc[["2011-09","2011-10","2011-11"],"Revenue"].sum()/mon.Revenue.sum()
R["nov_share"] = mon.loc["2011-11","Revenue"]/mon.Revenue.sum()
R["nov_vs_janaug"] = mon.loc["2011-11","Revenue"]/mon.loc["2011-01":"2011-08","Revenue"].mean()

cu = idn.groupby("CustomerID").Revenue.sum().sort_values(ascending=False)
R["top20_share"] = cu.head(int(len(cu)*.2)).sum()/cu.sum(); R["top1_share"] = cu.head(int(len(cu)*.01)).sum()/cu.sum()
oc = idn.groupby("CustomerID").InvoiceNo.nunique()
R["onetime_cust_share"] = (oc==1).mean(); R["onetime_rev_share"] = cu[oc[oc==1].index].sum()/cu.sum()
R["repeat_avg"] = cu[oc[oc>1].index].mean(); R["onetime_avg"] = cu[oc[oc==1].index].mean()

segs = rfm.groupby("Segment").agg(Customers=("mon","size"), Revenue=("mon","sum"), AvgRecencyDays=("recency","mean"), AvgOrders=("freq","mean"))
segs["RevShare"] = segs.Revenue/segs.Revenue.sum(); segs["CustShare"] = segs.Customers/segs.Customers.sum()
segs.to_csv(f"{OUT}/summary_segments.csv")

ctry = sales.groupby("Country").agg(Revenue=("Revenue","sum"), Customers=("CustomerID","nunique"))
ctry["Orders"] = inv.groupby("Country").size(); ctry["AOV"] = ctry.Revenue/ctry.Orders
ctry["Share"] = ctry.Revenue/ctry.Revenue.sum(); ctry = ctry.sort_values("Revenue", ascending=False); ctry.to_csv(f"{OUT}/summary_country.csv")
uk = inv[inv.Region=="UK"]; ex = inv[inv.Region=="Export"]
R.update(uk_aov=uk.Revenue.mean(), ex_aov=ex.Revenue.mean(), uk_med=float(uk.Revenue.median()), ex_med=float(ex.Revenue.median()),
         ex_rev_share=ex.Revenue.sum()/inv.Revenue.sum(),
         mw_p=float(mannwhitneyu(ex.Revenue, uk.Revenue, alternative="greater").pvalue))
ex_id = idn[idn.Country!="United Kingdom"]; ex_cust = ex_id.groupby("CustomerID").Revenue.sum().sort_values(ascending=False)
R["ex_cust_share_of_ident"] = ex_id.CustomerID.nunique()/idn.CustomerID.nunique()
R["ex_rev_share_of_ident"] = ex_id.Revenue.sum()/idn.Revenue.sum()
R["ex_top5_share"] = ex_cust.head(5).sum()/ex_cust.sum()
R["ex_per_cust_excl_top5"] = ex_cust.iloc[5:].mean(); R["uk_per_cust"] = idn[idn.Country=="United Kingdom"].groupby("CustomerID").Revenue.sum().mean()
R["nl_top_customer"] = float(ex_cust.iloc[0]); R["nl_top_share_total"] = ex_cust.iloc[0]/gross

wd = inv.groupby("Weekday").agg(Revenue=("Revenue","sum"), Orders=("InvoiceNo","count")); wd["AOV"] = wd.Revenue/wd.Orders
wd = wd.reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]); wd.to_csv(f"{OUT}/summary_weekday.csv")
hr = inv.groupby("Hour").Revenue.sum(); hr.to_csv(f"{OUT}/summary_hour.csv")
R["thu_share"] = wd.loc["Thursday","Revenue"]/wd.Revenue.sum(); R["sat_orders"] = int(wd.loc["Saturday","Orders"]) if not np.isnan(wd.loc["Saturday","Orders"]) else 0
R["peak_hours_share"] = hr.loc[10:15].sum()/hr.sum()

prod = sales.groupby("StockCode").agg(Description=("Description","first"), Revenue=("Revenue","sum"), Units=("Quantity","sum"), Orders=("InvoiceNo","nunique")).sort_values("Revenue", ascending=False)
prod["CumShare"] = prod.Revenue.cumsum()/prod.Revenue.sum(); prod.to_csv(f"{OUT}/summary_product.csv")
R["prod_top20_share"] = prod.head(int(len(prod)*.2)).Revenue.sum()/prod.Revenue.sum()
R["prod_n50"] = int((prod.CumShare<.5).sum()+1); R["prod_n80"] = int((prod.CumShare<.8).sum()+1)
R["large_orders_share_rev"] = inv[inv.Revenue>1000].Revenue.sum()/inv.Revenue.sum(); R["large_orders_share_n"] = (inv.Revenue>1000).mean()

rc = (-rets.groupby("Country").Revenue.sum()/ctry.Revenue).dropna().sort_values(ascending=False); rc.head(10).to_csv(f"{OUT}/summary_return_rate_country.csv")
R["usa_return_rate"] = float(rc.get("USA", np.nan))

json.dump({k: (float(v) if isinstance(v,(np.floating,float,np.integer)) else v) for k,v in R.items()}, open(f"{OUT}/key_numbers.json","w"), indent=1)

## 6. CHARTS

In [ ]:
# ---------- 6. CHARTS ----------
plt.rcParams.update({"font.family":"DejaVu Sans","axes.spines.top":False,"axes.spines.right":False})
fig, ax = plt.subplots(figsize=(9,4)); c = ["#B8B8B8"]*len(mon)
for i,m in enumerate(mon.index):
    if m in ("2011-09","2011-10","2011-11"): c[i] = "#1F7A8C"
ax.bar(mon.index, mon.Revenue/1e3, color=c); ax.set_ylabel("Revenue (GBP thousands)"); ax.set_title("Monthly revenue: Sep-Nov 2011 carries 36% of the year", loc="left")
ax.tick_params(axis="x", rotation=45); ax.annotate("Dec 2011: data ends 9 Dec", (len(mon)-1, mon.Revenue.iloc[-1]/1e3), xytext=(len(mon)-5.2, 1300), arrowprops=dict(arrowstyle="->"), fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/charts/monthly_revenue.png", dpi=160); plt.show()

fig, ax = plt.subplots(figsize=(7,4)); s = segs.sort_values("Revenue")
ax.barh(s.index, s.RevShare*100, color="#1F7A8C"); ax.set_xlabel("% of identified-customer revenue")
for i,(v,cc) in enumerate(zip(s.RevShare*100, s.CustShare*100)): ax.text(v+1, i, f"{v:.0f}% revenue from {cc:.0f}% of customers", va="center", fontsize=8)
ax.set_xlim(0,115); ax.set_title("Revenue by customer segment", loc="left"); plt.tight_layout(); plt.savefig(f"{OUT}/charts/segments.png", dpi=160); plt.show()

fig, ax = plt.subplots(figsize=(7,4)); lab = ["UK","Export (all)","Export excl. top 5 accounts"]
vals = [R["uk_per_cust"], ex_cust.mean(), R["ex_per_cust_excl_top5"]]
ax.bar(lab, vals, color=["#B8B8B8","#E07A5F","#1F7A8C"]); ax.set_ylabel("Avg revenue per customer (GBP)"); ax.set_title("Export value is driven by 5 accounts", loc="left")
for i,v in enumerate(vals): ax.text(i, v+60, f"GBP {v:,.0f}", ha="center", fontsize=9)
plt.tight_layout(); plt.savefig(f"{OUT}/charts/export_vs_uk.png", dpi=160); plt.show()

fig, ax = plt.subplots(figsize=(7,3.6)); ax.bar(wd.index.str[:3], wd.Revenue.fillna(0)/1e6, color=["#1F7A8C" if d=="Thursday" else "#B8B8B8" for d in wd.index])
ax.set_ylabel("Revenue (GBP millions)"); ax.set_title("Revenue by weekday (no Saturday trading recorded)", loc="left"); plt.tight_layout(); plt.savefig(f"{OUT}/charts/weekday.png", dpi=160); plt.show()
print(json.dumps(R, indent=1, default=float))

## Download results

In [ ]:
!zip -qr output.zip output
from google.colab import files
files.download("output.zip")